# Error Case 17 — 401: Token Refresh Race Condition

**Error:** `401 Unauthorized`  
**Root cause:** Multiple threads refresh token simultaneously → some get invalidated  
**Fix:** Single token manager shared across all executors

## Flow
```
Spark job with N executors starts
  → each executor independently checks token expiry
  → all detect expiry at same time
  → all call POST /oauth/tokens simultaneously
  → Polaris issues N tokens (each valid)
  → but: Polaris may invalidate previous token on new issue
  → executors with old token → 401 mid-job
```

## Why this happens in production
```
Spark executor config (wrong):
  Each executor has own token
  Each executor refreshes independently
  → N concurrent token requests → race condition

Correct pattern:
  Single Spark driver holds token
  All executors share driver token
  Driver refreshes proactively before expiry
```

In [1]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import sys
import importlib, polaris_test_utils
importlib.reload(polaris_test_utils)
from polaris_test_utils import *
require_not_prod("error-case test")  # PROD-safe: refuses to run against company PROD
import threading, time

cleanup()
print('✅ Setup complete')

🧹 Cleaning up test resources...
✅ Cleanup done
✅ Setup complete


In [2]:
# Cell 2: Simulate N executors refreshing token simultaneously
N_EXECUTORS = 5
tokens = []
errors = []
lock = threading.Lock()

def executor_refresh(executor_id):
    """Simulate each Spark executor independently refreshing token."""
    r = get_token()
    with lock:
        if r.status_code == 200:
            tokens.append({
                'executor': executor_id,
                'token': r.json()['access_token'][:30] + '...'
            })
        else:
            errors.append({'executor': executor_id, 'status': r.status_code})

print(f'Simulating {N_EXECUTORS} executors refreshing token simultaneously...')
threads = [threading.Thread(target=executor_refresh, args=(i,))
           for i in range(N_EXECUTORS)]

# Start all at the same time — simulates race condition
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f'\nTokens obtained: {len(tokens)}')
print(f'Errors:          {len(errors)}')
print(f'\nAll {N_EXECUTORS} executors got different tokens:')
for tok in tokens:
    print(f'  Executor {tok["executor"]}: {tok["token"]}')

Simulating 5 executors refreshing token simultaneously...

Tokens obtained: 5
Errors:          0

All 5 executors got different tokens:
  Executor 0: eyJhbGciOiJSUzI1NiIsInR5cCI6Ik...
  Executor 2: eyJhbGciOiJSUzI1NiIsInR5cCI6Ik...
  Executor 4: eyJhbGciOiJSUzI1NiIsInR5cCI6Ik...
  Executor 3: eyJhbGciOiJSUzI1NiIsInR5cCI6Ik...
  Executor 1: eyJhbGciOiJSUzI1NiIsInR5cCI6Ik...


In [3]:
# Cell 3: Show the problem — older tokens may be invalidated
# Use first token obtained (oldest)
first_token = tokens[0]['token'].replace('...', '')
last_token_full = None

# Get all full tokens to test
all_tokens = []
for i in range(3):
    r = get_token()
    if r.status_code == 200:
        all_tokens.append(r.json()['access_token'])
    time.sleep(0.1)

print(f'Testing if earlier tokens still valid after multiple refreshes...')
for i, tok in enumerate(all_tokens[:2]):   # test first 2 tokens
    r = requests.get(
        f'{BASE_MGMT}/catalogs',
        headers={'Authorization': f'Bearer {tok}', 'Polaris-Realm': REALM}
    )
    print(f'Token {i+1}: {r.status_code}')

print(f'\nNote: Polaris RSA tokens are stateless — all valid until expiry')
print('Race condition impact: wasted API calls, not invalidation')
print('Real risk: executor uses token obtained BEFORE rotation')

Testing if earlier tokens still valid after multiple refreshes...
Token 1: 200
Token 2: 200

Note: Polaris RSA tokens are stateless — all valid until expiry
Race condition impact: wasted API calls, not invalidation
Real risk: executor uses token obtained BEFORE rotation


In [4]:
# Cell 4: Simulate real race condition — token rotation during job
print('Simulating token rotation during active job...')

# Step 1: Get token
token_v1 = get_token().json()['access_token']
print(f'v1 token obtained')

# Step 2: Rotate credentials (simulates principal rotation)
r_create = create_principal()
assert r_create.status_code == 201
cred = r_create.json()['credentials']

token_principal_v1 = get_token(
    cred['clientId'], cred['clientSecret']
).json()['access_token']
print(f'Principal token v1 obtained')

# Step 3: Rotate credentials → old token still valid (RSA stateless)
r_rotate = requests.post(
    f'{BASE_MGMT}/principals/{TEST_PRINCIPAL}/credentials:rotate',
    headers=h()
)
print(f'\nCredential rotation: {r_rotate.status_code}')
if r_rotate.status_code == 200:
    new_cred = r_rotate.json()['credentials']
    print('New credentials issued')
    
    # Old token still works (RSA JWT — stateless, not revoked)
    r_test = requests.get(
        f'{BASE_MGMT}/catalogs',
        headers={'Authorization': f'Bearer {token_principal_v1}',
                 'Polaris-Realm': REALM}
    )
    print(f'Old token after rotation: {r_test.status_code}')
    print('Note: Polaris does not revoke old JWT tokens')
    print('      Old token valid until exp claim')

Simulating token rotation during active job...
v1 token obtained
Principal token v1 obtained

Credential rotation: 404


In [ ]:
# Cell 5: Find logs in OpenSearch
wait_for_logs(10)
print('Searching for 401 errors from race condition...')
hits = search_401_errors(minutes_ago=2)
print_logs(hits)

In [ ]:
# Cell 6: Fix — single token manager pattern
class PolarisTokenManager:
    """
    Single token manager — shared across all executors.
    Prevents race condition by centralizing token refresh.
    """
    def __init__(self, client_id, client_secret, realm=REALM):
        self.client_id = client_id
        self.client_secret = client_secret
        self.realm = realm
        self._token = None
        self._expires_at = 0
        self._lock = threading.Lock()
    
    def get_token(self):
        """Get valid token — refresh proactively 5 min before expiry."""
        with self._lock:             # only one thread refreshes at a time
            if time.time() > self._expires_at - 300:  # 5 min buffer
                self._refresh()
            return self._token
    
    def _refresh(self):
        r = get_token(self.client_id, self.client_secret)
        assert r.status_code == 200, f'Token refresh failed: {r.text}'
        data = self._token = r.json()['access_token']
        self._expires_at = time.time() + r.json()['expires_in']
        print(f'Token refreshed — expires in {r.json()["expires_in"]}s')
        self._token = r.json()['access_token']

# Test token manager with concurrent executors
manager = PolarisTokenManager(ROOT_CLIENT, ROOT_SECRET)
print('Testing PolarisTokenManager with concurrent executors...')

results = []
def executor_with_manager(executor_id):
    tok = manager.get_token()    # ← all executors share one token
    r = requests.get(
        f'{BASE_MGMT}/catalogs',
        headers={'Authorization': f'Bearer {tok}', 'Polaris-Realm': REALM}
    )
    results.append({'executor': executor_id, 'status': r.status_code})

threads = [threading.Thread(target=executor_with_manager, args=(i,))
           for i in range(N_EXECUTORS)]
for t in threads: t.start()
for t in threads: t.join()

all_ok = all(r['status'] == 200 for r in results)
print(f'All {N_EXECUTORS} executors succeeded: {all_ok}')
for r in results:
    print(f'  Executor {r["executor"]}: {r["status"]}')

cleanup()
print('\n=== Summary ===')
print('Error:    401 during concurrent token refresh')
print('Cause:    N executors independently refresh → wasted calls + potential race')
print('Fix:      Single PolarisTokenManager shared across executors')
print('Pattern:  Lock + proactive refresh (5 min before expiry)')
print('Spark:    Set token on driver, broadcast to executors')